Import Library

In [2]:
from pathlib import Path
import torch
import pandas as pd
import sys
import os

# go up two levels: Jupyter_notebook → scripts → project_root
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)


# reuse functions from your script
# from Archeived.main_predict import get_model, get_dataset, run_pred
from mst.inference.predictor import load_model, get_dataset_class, predict_batch


import warnings
warnings.simplefilter("ignore", UserWarning)

Set Device and Path

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# run_dir = Path("/home/jovyan/work/MST/runs")
# run_folder = Path("ODELIA/DinoV2ClassifierSlice_Final")
# path_run = run_dir / run_folder

##For New Model
run_folder = Path("NewModel")
checkpoint_name = "challenge_mstv3-vit_sch_CB_sub2_best.chkpt"
path_run = run_dir / run_folder / checkpoint_name

Load Model

In [7]:
model = load_model("DinoV2ClassifierSlice", path_run, device)
model.to(device)
model.eval()

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /home/jovyan/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /home/jovyan/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:00<00:00, 95.4MB/s]


DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
   

In [10]:
print(model.encoder)

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [11]:
print(model)

DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
   

Load Test set

In [41]:
ds_test = get_dataset_class(name="ODELIA")(split="test")

In [42]:
print(ds_test.__dict__.keys())

dict_keys(['path_root', 'path_root_data', 'split', 'sequence', 'transform', 'df', 'item_pointers'])


In [43]:
target_uid = "02F4A1FB_left"

idx = ds_test.item_pointers.index(target_uid)
sample = ds_test[idx]

In [44]:
batch = {}
for k, v in sample.items():
    if torch.is_tensor(v):
        batch[k] = v.unsqueeze(0).to(device)
    else:
        batch[k] = v


In [45]:
batch

{'uid': '02F4A1FB_left',
 'source': tensor([[[[[-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            ...,
            [-0.5868, -0.6288, -0.6288,  ...,  1.1775,  0.9464,  0.9464],
            [-0.5868, -0.6078, -0.6498,  ...,  0.9254,  0.8624,  0.9674],
            [-0.6078, -0.6078, -0.6288,  ...,  0.8414,  0.8414,  0.9884]],
 
           [[-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            ...,
            [-0.6078, -0.5448, -0.4818,  ...,  1.1565,  0.9044,  0.9044],
            [-0.5868, -0.5028, -0.4818,  ...,  0.9884,  0.7574,  0.7364],
            [-0.5028, -0.4397, -0.4818,  ...,  1.0514,  0.6314,  0.5264]],
 
           [[-0.6288, -0.6288, -0.628

In [46]:
batch["source"].shape

torch.Size([1, 1, 32, 224, 224])

In [47]:
pred = predict_batch(model, batch, device=device).cpu()
#probs = torch.softmax(pred, dim=-1).cpu()
pred_class = torch.argmax(pred, dim=1)

print(f"Pred is {pred}")
print(f"Predicted class is {pred_class}")

print("GT:", batch["target"].item())
print("Predicted class:", pred_class.item())
print("Probabilities:", pred.squeeze().numpy())


Pred is tensor([[4.0924e-01, 5.5058e-04, 5.9021e-01]])
Predicted class is tensor([2])
GT: 0
Predicted class: 2
Probabilities: [4.092439e-01 5.505763e-04 5.902055e-01]


In [48]:
with torch.no_grad():
    logits = model(batch["source"].cpu())
    prob = torch.softmax(logits, dim=-1).cpu()#[0]
predicted_class = logits.argmax(dim=-1).item()

print(f"Logits: {logits}")
print(f"Probabilities: {prob}")
print(f"Predicted class from logits: {predicted_class}")

Logits: tensor([[ 2.1233, -4.4878,  2.4894]], device='cuda:0')
Probabilities: tensor([[4.0924e-01, 5.5058e-04, 5.9021e-01]])
Predicted class from logits: 2


In [49]:
prob.detach().cpu().tolist()

[[0.4092439115047455, 0.000550576311070472, 0.5902054905891418]]

In [50]:
model.eval()

with torch.no_grad():
    logits_predict = model(
        batch["source"],
        src_key_padding_mask=batch.get("src_key_padding_mask")
    )

    # simulate perturbation step 0
    current = batch["source"].clone()

    logits_pert0 = model(
        current,
        src_key_padding_mask=batch.get("src_key_padding_mask")
    )

print(torch.allclose(logits_predict, logits_pert0, atol=1e-6))
print((logits_predict - logits_pert0).abs().max())

True
tensor(0., device='cuda:0')


In [51]:
print(batch["source"].shape)
print(current.shape)
print(torch.equal(batch["source"], current))

torch.Size([1, 1, 32, 224, 224])
torch.Size([1, 1, 32, 224, 224])
True


In [ ]:
print(batch.get("src_key_padding_mask"))